In [ ]:
import shutil
import subprocess
import time
from datetime import datetime
from pathlib import Path

import pandas as pd

GENERATED_ROOT = Path("/data/home/mirick/ChordEdit/daniel-grid-optimizations/daniel_grid_optimizations/generated/UltraEdit_Region_10")
SCRIPT_DIR = Path("/data/home/mirick/ChordEdit/daniel-grid-optimizations/daniel_grid_optimizations")
RESULT_CSV = GENERATED_ROOT / "id_to_metrics_ultraeditregion10.csv"

BENCH_DIR = Path("/data/home/mirick/ChordEdit/daniel-grid-optimizations/bench_results")
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
BENCH_DIR.mkdir(parents=True, exist_ok=True)

print(f"Bench dir:  {BENCH_DIR}")
print(f"Timestamp:  {TIMESTAMP}")
print(f"Result CSV: {RESULT_CSV}")

## Run optimized `grid_eval.py`

In [ ]:
t0 = time.perf_counter()
result_opt = subprocess.run(
    ["python", "grid_eval.py", "--generated-root", str(GENERATED_ROOT)],
    cwd=str(SCRIPT_DIR),
    capture_output=True, text=True,
)
elapsed_opt = time.perf_counter() - t0

print(result_opt.stderr)
if result_opt.returncode != 0:
    raise RuntimeError(f"Optimized run failed (exit {result_opt.returncode})")

opt_csv = BENCH_DIR / f"metrics_optimized_{TIMESTAMP}.csv"
shutil.copy2(RESULT_CSV, opt_csv)
print(f"\nSaved to {opt_csv}")
print(f"Elapsed: {elapsed_opt:.2f}s")

## Run original `grid_eval.py.bak`

In [ ]:
t0 = time.perf_counter()
result_bak = subprocess.run(
    ["python", "grid_eval.py.bak", "--generated-root", str(GENERATED_ROOT)],
    cwd=str(SCRIPT_DIR),
    capture_output=True, text=True,
)
elapsed_bak = time.perf_counter() - t0

print(result_bak.stderr)
if result_bak.returncode != 0:
    raise RuntimeError(f"Original run failed (exit {result_bak.returncode})")

bak_csv = BENCH_DIR / f"metrics_original_{TIMESTAMP}.csv"
shutil.copy2(RESULT_CSV, bak_csv)
print(f"\nSaved to {bak_csv}")
print(f"Elapsed: {elapsed_bak:.2f}s")

## Compare outputs

In [ ]:
df_opt = pd.read_csv(opt_csv)
df_bak = pd.read_csv(bak_csv)

print(f"Optimized rows: {len(df_opt)}")
print(f"Original  rows: {len(df_bak)}")

# Align on the same key columns
key_cols = ["sample_id", "t_start", "t_end"]
metric_cols = ["psnr_unedit_part", "lpips_unedit_part", "clip_similarity_target_image_edit_part"]

df_merged = df_opt.merge(df_bak, on=key_cols, suffixes=("_opt", "_bak"))
print(f"Matched rows:   {len(df_merged)}")

# Per-metric absolute differences
for col in metric_cols:
    opt_vals = pd.to_numeric(df_merged[f"{col}_opt"], errors="coerce")
    bak_vals = pd.to_numeric(df_merged[f"{col}_bak"], errors="coerce")
    diff = (opt_vals - bak_vals).abs()
    print(f"\n--- {col} ---")
    print(f"  max  abs diff: {diff.max():.8f}")
    print(f"  mean abs diff: {diff.mean():.8f}")
    print(f"  rows with >0.01 diff: {(diff > 0.01).sum()}")

## Timing summary

In [ ]:
speedup = elapsed_bak / elapsed_opt if elapsed_opt > 0 else float("inf")

summary = pd.DataFrame({
    "version": ["original (grid_eval.py.bak)", "optimized (grid_eval.py)"],
    "elapsed_s": [f"{elapsed_bak:.2f}", f"{elapsed_opt:.2f}"],
    "rows": [len(df_bak), len(df_opt)],
})
display(summary)
print(f"\nSpeedup: {speedup:.2f}x")

# Save summary
summary_path = BENCH_DIR / f"bench_summary_{TIMESTAMP}.csv"
summary.to_csv(summary_path, index=False)
print(f"Summary saved to {summary_path}")